# 🕸️ Abordagem B1 — Knowledge Graph via NER Puro (sem Ontologia)

## Módulo B: Knowledge Graph como Base de Recuperação

Este notebook implementa a **Abordagem B1** do experimento comparativo de estratégias RAG avançadas.

### Posicionamento no Experimento

```
Corpus Wikipedia
       │
       └── Módulo B: Knowledge Graph ────┬── B1. NER puro (sem ontologia)  ← VOCÊ ESTÁ AQUI
                                         ├── B2. Ontologia pré-definida + extração
                                         └── B3. Estrutura semântica (comunidades)
```

### O que é GraphRAG?

Nas abordagens do **Módulo A** (chunking vetorial), a recuperação é feita por *similaridade semântica* — o sistema encontra pedaços de texto cujos vetores são mais próximos do vetor da pergunta. Isso funciona bem para perguntas diretas, mas tem limitações quando a resposta exige **cruzar informações de múltiplas fontes** (perguntas *multi-hop*).

O **GraphRAG** substitui o banco vetorial por um **grafo de conhecimento**:

```
Documentos Wikipedia
        │
   Extração de ──► Entidades (nós) + Relações (arestas)
   Conhecimento
        │
        ▼
   Grafo de Conhecimento (NetworkX)
        │
   Query → Identificar entidades via NER
        │
   Traversal → Expandir k saltos a partir das entidades
        │
   Recuperar textos das arestas do subgrafo
        │
   Gerador LLM (mesmo das abordagens A)
```

### Por que B1 é o ponto de partida?

A Abordagem B1 é a versão **mais simples e exploratória** do Knowledge Graph:

| Característica | B1 (este notebook) | B2 (próximo) | B3 (depois) |
|----------------|-------------------|--------------|-------------|
| Tipo de relação | `CO_OCORRE_COM` (co-ocorrência bruta) | Relações ontológicas | Comunidades semânticas |
| Custo de indexação | **$0.00** (só spaCy local) | ~$0.02 (LLM) | ~$0.024 (LLM) |
| Precisão esperada | **Baixa-Moderada** | Alta | Alta (global) |

### O que observar nesta abordagem

- **Contextual Precision tende a ser baixa**: arestas `CO_OCORRE_COM` são ruidosas
  - B1: `"Cameron" --CO_OCORRE_COM--> "1998"` (relação vaga)
  - B2: `"James Cameron" --PREMIADO_COM--> "Oscar de Melhor Diretor"` (semântico)
- **Multi-hop naturalmente suportado**: traversal com k_hops=2 encadeia entidades
- **Custo zero de indexação**: spaCy roda localmente, sem chamada de API
- **Hipótese H4** a verificar: KG sem ontologia tem Contextual Precision inferior ao B2?

In [ ]:
# ============================================================
# CÉLULA 2 — Configuração do Ambiente e Carregamento dos Dados
# ============================================================

# Instalação das dependências do Módulo B (execute apenas uma vez):
# %pip install spacy networkx pyvis langchain langchain-openai langfuse datasets pandas deepeval python-dotenv
# %python -m spacy download en_core_web_trf

import os
import pickle
import time
from pathlib import Path
from typing import List, Dict, Tuple

import pandas as pd
import networkx as nx
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️  AVISO: OPENAI_API_KEY não encontrada. Verifique o arquivo .env!")
if not os.environ.get("LANGFUSE_PUBLIC_KEY"):
    print("⚠️  AVISO: Credenciais do Langfuse não encontradas. Verifique o arquivo .env!")

# -------------------------------------------------------
# Dataset didático inline (usado quando HuggingFace indisponível)
# -------------------------------------------------------
DATASET_INLINE = {
    "questoes": [
        {"id": "demo_001", "query": "Who directed Titanic and what award did it win?",
         "golden_doc": ["James Cameron", "Titanic film"],
         "ground_truth": "James Cameron directed Titanic, which won the Academy Award for Best Picture in 1998.",
         "tipo": "bridge", "dataset": "demo"},
        {"id": "demo_002", "query": "Where was the director of Avatar born?",
         "golden_doc": ["James Cameron", "Kapuskasing"],
         "ground_truth": "James Cameron was born in Kapuskasing, Ontario, Canada.",
         "tipo": "bridge", "dataset": "demo"},
        {"id": "demo_003", "query": "Who is older, James Cameron or Steven Spielberg?",
         "golden_doc": ["James Cameron", "Steven Spielberg"],
         "ground_truth": "Steven Spielberg is older. He was born in 1946; Cameron in 1954.",
         "tipo": "comparison", "dataset": "demo"},
        {"id": "demo_004", "query": "In what country is the city where the Eiffel Tower is located?",
         "golden_doc": ["Eiffel Tower", "Paris"],
         "ground_truth": "The Eiffel Tower is in Paris, which is in France.",
         "tipo": "inference", "dataset": "demo"},
        {"id": "demo_005", "query": "Were Marie Curie and Albert Einstein from the same country?",
         "golden_doc": ["Marie Curie", "Albert Einstein"],
         "ground_truth": "No. Marie Curie was Polish (later French); Einstein was German (later American).",
         "tipo": "comparison", "dataset": "demo"},
    ],
    "documentos": [
        {"id": "James Cameron", "titulo": "James Cameron", "texto":
            "James Francis Cameron was born on August 16, 1954, in Kapuskasing, Ontario, Canada. "
            "He is a Canadian filmmaker and director. Cameron wrote and directed The Terminator (1984), "
            "Aliens (1986), Titanic (1997), and Avatar (2009). "
            "Titanic earned him Academy Awards for Best Picture and Best Director in 1998. "
            "Avatar became the highest-grossing film of all time upon its release in 2009. "
            "Cameron has received numerous awards including the Academy Award for Best Director."},
        {"id": "Titanic film", "titulo": "Titanic (1997 film)", "texto":
            "Titanic is a 1997 American epic romance and disaster film directed by James Cameron. "
            "It stars Leonardo DiCaprio and Kate Winslet. The film was released in December 1997. "
            "At the 70th Academy Awards, Titanic won eleven Academy Awards including Best Picture and Best Director. "
            "James Cameron wrote and co-produced the film. The film grossed over $2 billion worldwide."},
        {"id": "Avatar film", "titulo": "Avatar (2009 film)", "texto":
            "Avatar is a 2009 science fiction film written and directed by James Cameron. "
            "It stars Sam Worthington, Zoe Saldana, and Sigourney Weaver. "
            "Avatar surpassed Titanic to become the highest-grossing film of all time. "
            "James Cameron won a Golden Globe Award for Best Director for Avatar."},
        {"id": "Steven Spielberg", "titulo": "Steven Spielberg", "texto":
            "Steven Allan Spielberg was born on December 18, 1946, in Cincinnati, Ohio, USA. "
            "He is an American film director and producer. Spielberg directed Jaws (1975), "
            "E.T. the Extra-Terrestrial (1982), Schindler's List (1993), and Saving Private Ryan (1998). "
            "Schindler's List won the Academy Award for Best Picture and Best Director. "
            "Spielberg has won three Academy Awards."},
        {"id": "Eiffel Tower", "titulo": "Eiffel Tower", "texto":
            "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. "
            "It is named after the engineer Gustave Eiffel. Constructed from 1887 to 1889. "
            "The Eiffel Tower stands 330 metres tall. "
            "It is the most-visited paid monument in the world."},
        {"id": "Paris", "titulo": "Paris", "texto":
            "Paris is the capital and most populous city of France. "
            "It is located in north-central France along the Seine River. "
            "Paris is home to famous landmarks including the Eiffel Tower and the Louvre Museum. "
            "Paris has been a major center of art, culture, and politics for centuries."},
        {"id": "Marie Curie", "titulo": "Marie Curie", "texto":
            "Marie Salomea Curie was born Maria Sklodowska on November 7, 1867, in Warsaw, Poland. "
            "She was a Polish and naturalized-French physicist and chemist. "
            "Curie conducted pioneering research on radioactivity. "
            "She was the first woman to win a Nobel Prize: Physics in 1903 and Chemistry in 1911. "
            "Marie Curie moved to Paris, France to study at the University of Paris."},
        {"id": "Albert Einstein", "titulo": "Albert Einstein", "texto":
            "Albert Einstein was born on March 14, 1879, in Ulm, Kingdom of Wurttemberg, Germany. "
            "He was a German-born theoretical physicist who developed the theory of relativity. "
            "Einstein received the Nobel Prize in Physics in 1921. "
            "He moved to the United States in 1933 when Adolf Hitler came to power in Germany. "
            "Einstein worked at the Institute for Advanced Study in Princeton, New Jersey."},
        {"id": "Kapuskasing", "titulo": "Kapuskasing", "texto":
            "Kapuskasing is a town in Cochrane District in northeastern Ontario, Canada. "
            "It is located on the Kapuskasing River, a tributary of the Mattagami River. "
            "The town has a population of approximately 8,000 residents. "
            "Kapuskasing is known as the birthplace of filmmaker James Cameron."},
        {"id": "Academy Awards", "titulo": "Academy Awards", "texto":
            "The Academy Awards, also known as the Oscars, are awards for merit in film. "
            "They are presented annually by the Academy of Motion Picture Arts and Sciences. "
            "The Best Picture award is the most prestigious award in the film industry. "
            "Titanic and Ben-Hur hold the record of most Oscar wins for a single film, with 11 each."},
    ]
}


def carregar_ou_criar_subset() -> Tuple[List[dict], List[dict]]:
    """
    Carrega o subset de questões gerado pelo notebook 00_exploracao_dados.ipynb.
    Se não existir, tenta HuggingFace. Se falhar, usa dataset didático inline.

    Retorna:
        questoes: lista de dicts com {id, query, golden_doc, ground_truth, tipo, dataset}
        documentos: lista de dicts com {id, titulo, texto}
    """
    # 1. Tentar subset pré-gerado
    caminho = Path("subset_100q.pkl")
    if caminho.exists():
        print("✅ Carregando subset pré-gerado de subset_100q.pkl...")
        with open(caminho, "rb") as f:
            dados = pickle.load(f)
        return dados["questoes"], dados["documentos"]

    print("⏳ subset_100q.pkl não encontrado. Tentando HuggingFace...")

    # 2. Tentar HuggingFace
    try:
        from datasets import load_dataset

        print("📥 Baixando 2wiki.jsonl (50 questões)...")
        ds_2wiki = load_dataset(
            "AQ-MedAI/RAG-QA-Leaderboard",
            data_files={"train": "final_data/2wiki.jsonl"},
            split="train[:50]"
        )
        print("📥 Baixando hotpot_distractor.jsonl (50 questões)...")
        ds_hotpot = load_dataset(
            "AQ-MedAI/RAG-QA-Leaderboard",
            data_files={"train": "final_data/hotpot_distractor.jsonl"},
            split="train[:50]"
        )

        questoes, documentos, doc_ids_vistos = [], [], set()

        for item in ds_2wiki:
            questoes.append({
                "id": item.get("id", f"2wiki_{len(questoes)}"),
                "query": item.get("query", item.get("question", "")),
                "golden_doc": item.get("golden_doc", []),
                "ground_truth": item.get("ground_truth", item.get("answer", "")),
                "tipo": item.get("type", "bridge"),
                "dataset": "2wiki"
            })
            for doc in item.get("context", []):
                doc_id = doc[0] if isinstance(doc, list) else doc.get("id", "")
                if doc_id not in doc_ids_vistos:
                    doc_ids_vistos.add(doc_id)
                    texto = " ".join(doc[1]) if isinstance(doc, list) else doc.get("text", "")
                    documentos.append({"id": doc_id, "titulo": doc_id, "texto": texto})

        for item in ds_hotpot:
            questoes.append({
                "id": item.get("id", f"hotpot_{len(questoes)}"),
                "query": item.get("question", ""),
                "golden_doc": [s[0] for s in item.get("supporting_facts", [])],
                "ground_truth": item.get("answer", ""),
                "tipo": "distractor",
                "dataset": "hotpot"
            })
            for doc in item.get("context", []):
                doc_id = doc[0] if isinstance(doc, list) else doc.get("id", "")
                if doc_id not in doc_ids_vistos:
                    doc_ids_vistos.add(doc_id)
                    texto = " ".join(doc[1]) if isinstance(doc, list) else doc.get("text", "")
                    documentos.append({"id": doc_id, "titulo": doc_id, "texto": texto})

        print(f"✅ {len(questoes)} questões e {len(documentos)} documentos carregados do HuggingFace")
        return questoes, documentos

    except Exception as e:
        print(f"⚠️  HuggingFace indisponível ({e}). Usando dataset didático inline.")
        return DATASET_INLINE["questoes"], DATASET_INLINE["documentos"]


questoes, documentos = carregar_ou_criar_subset()

print(f"\n📊 Dataset carregado:")
print(f"   • Questões:   {len(questoes)}")
print(f"   • Documentos: {len(documentos)}")

tipos = pd.Series([q.get("tipo", "?") for q in questoes]).value_counts()
print(f"\n   Distribuição por tipo de questão:")
for tipo, cnt in tipos.items():
    print(f"     - {tipo}: {cnt}")

print(f"\n   Exemplo: {questoes[0]['query']}")


In [ ]:
# ============================================================
# CÉLULA 3 — Indexação: Construção do Grafo de Conhecimento via NER
# ============================================================
# ETAPA DE INDEXAÇÃO da Abordagem B1.
#
# Diferença fundamental em relação ao Módulo A:
#   Módulo A: texto → chunks → embeddings → ChromaDB (similaridade vetorial)
#   Módulo B: texto → entidades NER → co-ocorrência → NetworkX DiGraph
#
# Grafo resultante:
#   NÓS   = entidades nomeadas (PERSON, ORG, GPE, DATE, WORK_OF_ART...)
#   ARESTAS = "CO_OCORRE_COM" — dois nós ligados se aparecem na mesma sentença
# ============================================================

import spacy

# ── 1. Carregar modelo spaCy ────────────────────────────────────────────
print("⏳ Carregando modelo spaCy para NER...")
try:
    nlp = spacy.load("en_core_web_trf")
    print("✅ Modelo en_core_web_trf carregado (precisão transformer)")
except OSError:
    nlp = spacy.load("en_core_web_sm")
    print("⚠️  Usando en_core_web_sm (fallback — menor precisão)")


# ── 2. Extração de triplas de co-ocorrência ─────────────────────────────

def extrair_entidades_e_relacoes(texto: str, doc_id: str) -> List[dict]:
    """
    Extrai triplas de co-ocorrência por sentença usando spaCy NER.

    Estratégia B1 — co-ocorrência simples:
    - NÓ: cada entidade nomeada reconhecida pelo NER
    - ARESTA: par de entidades que aparecem na MESMA sentença → CO_OCORRE_COM

    Limitação didática intencional comparada ao B2:
    - B1: "Cameron" --CO_OCORRE_COM--> "Oscar"  (relação vaga)
    - B2: "James Cameron" --PREMIADO_COM--> "Oscar de Melhor Diretor" (semântico)

    Args:
        texto:  conteúdo textual do documento
        doc_id: identificador único do documento de origem
    Returns:
        lista de triplas com campos: sujeito, tipo_sujeito, relacao,
                                     objeto, tipo_objeto, contexto, doc_id
    """
    doc = nlp(texto)
    triplas = []

    for sent in doc.sents:
        # Entidades únicas nesta sentença (filtrar tokens muito curtos = ruído)
        ents = [(e.text.strip(), e.label_) for e in sent.ents if len(e.text.strip()) > 1]
        ents = list(dict.fromkeys(ents))  # remover duplicatas mantendo ordem

        # Criar par para cada combinação de entidades na mesma sentença
        for i, (ent_a, tipo_a) in enumerate(ents):
            for ent_b, tipo_b in ents[i + 1:]:
                triplas.append({
                    "sujeito":      ent_a,
                    "tipo_sujeito": tipo_a,
                    "relacao":      "CO_OCORRE_COM",
                    "objeto":       ent_b,
                    "tipo_objeto":  tipo_b,
                    "contexto":     sent.text.strip(),
                    "doc_id":       doc_id
                })
    return triplas


# ── 3. Construção do grafo NetworkX DiGraph ──────────────────────────────

def construir_grafo_ner(documentos: List[dict]) -> nx.DiGraph:
    """
    Constrói o grafo de conhecimento NetworkX a partir dos documentos.

    Estrutura do grafo:
    - Nós:    entidades com atributos 'tipo' (label NER) e 'frequencia'
    - Arestas: co-ocorrências com atributos:
        relacao   → sempre "CO_OCORRE_COM" em B1
        contexto  → sentença de origem (primeira observada)
        contextos → lista de todas as sentenças com essa co-ocorrência
        doc_id    → documento de origem
        peso      → número de co-ocorrências (incrementa a cada nova observação)

    Args:
        documentos: lista de dicts {id, titulo, texto}
    Returns:
        G: DiGraph NetworkX populado
    """
    G = nx.DiGraph()
    total_triplas = 0
    inicio = time.time()

    print(f"🔨 Indexando {len(documentos)} documentos no grafo NER...")

    for i, doc in enumerate(documentos):
        if (i + 1) % 10 == 0 or i == 0:
            print(f"   [{i+1}/{len(documentos)}] {doc['titulo'][:50]}...")

        triplas = extrair_entidades_e_relacoes(doc["texto"], doc["id"])
        total_triplas += len(triplas)

        for t in triplas:
            # Adicionar nós com atributo tipo
            for no, tipo in [(t["sujeito"], t["tipo_sujeito"]),
                              (t["objeto"],  t["tipo_objeto"])]:
                if not G.has_node(no):
                    G.add_node(no, tipo=tipo, frequencia=0)
                G.nodes[no]["frequencia"] = G.nodes[no].get("frequencia", 0) + 1

            # Adicionar ou incrementar aresta (acumular peso e contextos)
            if G.has_edge(t["sujeito"], t["objeto"]):
                G[t["sujeito"]][t["objeto"]]["peso"] += 1
                G[t["sujeito"]][t["objeto"]]["contextos"].append(t["contexto"])
            else:
                G.add_edge(
                    t["sujeito"], t["objeto"],
                    relacao="CO_OCORRE_COM",
                    contexto=t["contexto"],
                    contextos=[t["contexto"]],
                    doc_id=t["doc_id"],
                    peso=1
                )

    duracao = time.time() - inicio
    print(f"\n✅ Grafo construído em {duracao:.1f}s")
    print(f"   Triplas de co-ocorrência processadas: {total_triplas:,}")
    return G


# ── 4. Executar indexação ────────────────────────────────────────────────
inicio_indexacao = time.time()
G_b1 = construir_grafo_ner(documentos)
tempo_indexacao_s = time.time() - inicio_indexacao

print(f"\n⏱️  Tempo total de indexação: {tempo_indexacao_s:.1f}s")
print(f"💰 Custo de indexação: $0.00 (spaCy é local, sem chamada de API)")


## 📊 Estrutura do Grafo Construído

Antes de recuperar informações, analisamos a estrutura do grafo. Esta análise é fundamental para o diagnóstico pedagógico da Abordagem B1.

### Por que analisar o grafo?

O grafo de co-ocorrência tem características distintas de um grafo com semântica explícita:

- **Alta densidade nos hubs**: nomes de pessoas famosas e datas se conectam com muitas entidades
- **Ruído estrutural**: `"Cameron"` e `"1998"` co-ocorrem em várias frases com motivações diferentes
- **Componentes desconexos**: entidades de documentos isolados podem não estar conectadas

As estatísticas abaixo revelam se o grafo é adequado para recuperação multi-hop.

In [ ]:
# ============================================================
# CÉLULA 4b — Análise Estatística do Grafo NER
# ============================================================

print("=" * 60)
print("ESTATÍSTICAS DO GRAFO B1 — Knowledge Graph NER")
print("=" * 60)

num_nos     = G_b1.number_of_nodes()
num_arestas = G_b1.number_of_edges()
densidade   = nx.density(G_b1)

print(f"\n📌 Estrutura Geral:")
print(f"   • Nós (entidades únicas):   {num_nos:,}")
print(f"   • Arestas (co-ocorrências): {num_arestas:,}")
print(f"   • Densidade do grafo:        {densidade:.6f}")
print(f"   • Razão arestas/nós:         {num_arestas / max(num_nos, 1):.2f}")

# Distribuição de tipos de entidade NER
tipos_nos = [d.get("tipo", "?") for _, d in G_b1.nodes(data=True)]
df_tipos  = pd.Series(tipos_nos).value_counts()
max_cnt   = df_tipos.max() if len(df_tipos) > 0 else 1
print(f"\n🏷️  Tipos de entidade NER:")
for tipo, cnt in df_tipos.items():
    barra = "█" * int(cnt / max_cnt * 25)
    print(f"   {tipo:<15} {barra:<27} {cnt}")

# Top 10 entidades mais conectadas (hubs)
graus   = dict(G_b1.degree())
top_nos = sorted(graus.items(), key=lambda x: x[1], reverse=True)[:10]
print(f"\n🔗 Top 10 entidades mais conectadas (hubs do grafo):")
for entidade, grau in top_nos:
    tipo = G_b1.nodes[entidade].get("tipo", "?")
    print(f"   [{tipo:<15}] {entidade:<35} grau: {grau}")

# Componentes conectados
G_und       = G_b1.to_undirected()
componentes = list(nx.connected_components(G_und))
tam_maior   = max((len(c) for c in componentes), default=0)
isolados    = sum(1 for c in componentes if len(c) == 1)
print(f"\n🔀 Componentes Conectados:")
print(f"   • Total:            {len(componentes)}")
print(f"   • Maior componente: {tam_maior} nós")
print(f"   • Nós isolados:     {isolados}")

# Distribuição de pesos
pesos = [d.get("peso", 1) for _, _, d in G_b1.edges(data=True)]
if pesos:
    print(f"\n⚖️  Pesos das arestas (co-ocorrências por par):")
    print(f"   • Mínimo: {min(pesos)}")
    print(f"   • Médio:  {sum(pesos)/len(pesos):.2f}")
    print(f"   • Máximo: {max(pesos)}")
    unicas = sum(1 for p in pesos if p == 1)
    print(f"   • Observadas apenas 1x: {unicas} ({100*unicas/len(pesos):.0f}%)")
    print()
    print("   ⚠️  Alta % de arestas peso=1 indica BAIXA repetição de co-ocorrência.")
    print("       Isso contribui para Contextual Precision baixa (hipótese H4).")

print(f"\n⏱️  Tempo de indexação: {tempo_indexacao_s:.1f}s | Custo: $0.00")


In [ ]:
# ============================================================
# CÉLULA 5 — Teste Manual de Recuperação (3 Exemplos)
# ============================================================
# Valida a função de recuperação antes de escalar para todas as questões.
# ============================================================


def recuperar_via_grafo_ner(
    query:         str,
    G:             nx.DiGraph,
    k_hops:        int = 2,
    max_contextos: int = 10
) -> List[Dict]:
    """
    Recupera contextos relevantes para uma query via traversal do grafo NER.

    Algoritmo:
    1. NER na query → identificar entidades mencionadas
    2. Localizar essas entidades como nós de entrada no grafo
       (matching exato + parcial para variações de nome)
    3. BFS limitada a k saltos a partir de cada nó de entrada
    4. Coletar textos das arestas no subgrafo resultante
    5. Ordenar por peso (arestas com mais co-ocorrências primeiro)

    Limitação do B1:
    Se a query menciona entidade não presente no grafo, nenhum contexto
    é recuperado para ela. O B2 mitiga isso com matching ontológico mais robusto.

    Args:
        query:         pergunta do usuário em linguagem natural
        G:             grafo construído na indexação
        k_hops:        profundidade BFS (2 = vizinhos dos vizinhos)
        max_contextos: limite de contextos a retornar
    Returns:
        lista de dicts {contexto, entidade_origem, entidade_destino, peso, doc_id}
    """
    # Passo 1: NER na query
    doc_query = nlp(query)
    entidades_query = [e.text.strip() for e in doc_query.ents if len(e.text.strip()) > 1]

    if not entidades_query:
        # Fallback: substantivos próprios
        entidades_query = [t.text for t in doc_query if t.pos_ == "PROPN"]

    print(f"   🔍 Entidades detectadas na query: {entidades_query}")

    # Passo 2: Encontrar nós no grafo (matching exato + parcial)
    nos_entrada = set()
    for ent in entidades_query:
        if ent in G:
            nos_entrada.add(ent)
        else:
            for no in G.nodes():
                if ent.lower() in no.lower() or no.lower() in ent.lower():
                    nos_entrada.add(no)

    print(f"   📍 Nós de entrada no grafo: {list(nos_entrada)[:5]}")

    if not nos_entrada:
        print(f"   ❌ Nenhum nó encontrado no grafo!")
        return []

    # Passo 3: Expansão BFS com k saltos
    subgrafo_nos = set()
    for no in nos_entrada:
        try:
            viz = nx.single_source_shortest_path_length(G, no, cutoff=k_hops)
            subgrafo_nos.update(viz.keys())
        except Exception:
            subgrafo_nos.add(no)

    print(f"   🕸️  Subgrafo expandido: {len(subgrafo_nos)} nós (k_hops={k_hops})")

    # Passo 4: Coletar contextos das arestas do subgrafo
    contextos = []
    for u, v, dados in G.edges(data=True):
        if u in subgrafo_nos or v in subgrafo_nos:
            # Prioridade extra quando AMBOS os nós pertencem ao subgrafo
            prioridade = 2 if (u in subgrafo_nos and v in subgrafo_nos) else 1
            contextos.append({
                "contexto":        dados.get("contexto", ""),
                "entidade_origem": u,
                "entidade_destino": v,
                "peso":            dados.get("peso", 1) * prioridade,
                "doc_id":          dados.get("doc_id", ""),
                "relacao":         "CO_OCORRE_COM"
            })

    # Passo 5: Ordenar por peso e desduplicar
    contextos.sort(key=lambda x: x["peso"], reverse=True)
    vistos, unicos = set(), []
    for item in contextos:
        if item["contexto"] not in vistos:
            vistos.add(item["contexto"])
            unicos.append(item)

    return unicos[:max_contextos]


# ── Executar testes manuais ──────────────────────────────────────────────
exemplos = questoes[:3] if len(questoes) >= 3 else questoes

print("=" * 70)
print("TESTE MANUAL DE RECUPERAÇÃO — 3 EXEMPLOS")
print("=" * 70)

for i, ex in enumerate(exemplos, 1):
    print(f"\n{'─'*70}")
    print(f"📌 Exemplo {i} | Tipo: {ex.get('tipo','?')}")
    print(f"❓ Query:        {ex['query']}")
    print(f"✅ Ground Truth: {ex['ground_truth'][:80]}...")
    print()

    resultados = recuperar_via_grafo_ner(ex["query"], G_b1, k_hops=2, max_contextos=5)

    if resultados:
        print(f"   📄 Top {len(resultados)} contextos recuperados:")
        for j, r in enumerate(resultados, 1):
            print(f"\n   [{j}] Doc: {r['doc_id']}")
            print(f"       {r['entidade_origem'][:20]} --CO_OCORRE_COM--> {r['entidade_destino'][:20]}")
            print(f"       Contexto: \"{r['contexto'][:100]}...\"")
            print(f"       Peso: {r['peso']}")
    else:
        print("   ❌ Nenhum contexto recuperado!")

    golden  = ex.get("golden_doc", [])
    doc_rec = {r["doc_id"] for r in resultados}
    cob     = len(set(golden) & doc_rec)
    print(f"\n   📊 Cobertura golden_docs: {cob}/{len(golden)}")


In [ ]:
# ============================================================
# CÉLULA 6 — Pipeline RAG Completo nas Questões do Subset
# ============================================================
# Executa recuperação + geração para cada questão,
# com rastreabilidade completa via Langfuse.
#
# Hierarquia no dashboard Langfuse:
#   Trace (B1_Pipeline_RAG_GraphNER)
#     └── Span       (B1_GraphRAG_Recuperacao) — traversal do grafo
#     └── Generation (B1_GraphRAG_Geracao)     — chamada LLM (tokens + custo)
# ============================================================

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langfuse import observe, get_client
from langfuse.langchain import CallbackHandler

# Inicializar Langfuse
langfuse_client  = get_client()
handler_langfuse = CallbackHandler()

# Modelo gerador idêntico às abordagens A (comparação justa)
modelo_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

# Prompt restritivo — mesmo princípio da Prática 1-3
PROMPT_RAG = ChatPromptTemplate.from_messages([
    ("system",
     """Você é um analista de informações preciso e criterioso.

Responda à pergunta usando EXCLUSIVAMENTE as evidências fornecidas abaixo.
Se a resposta não puder ser encontrada nas evidências, declare:
'Informação insuficiente nas evidências fornecidas.'

EVIDÊNCIAS:
{evidencias}"""),
    ("user", "Pergunta: {query}")
])


@observe(as_type="span", name="B1_GraphRAG_Recuperacao")
def recuperar_b1(query: str) -> Tuple[List[Dict], float]:
    """Recuperação via grafo NER instrumentada como Span no Langfuse."""
    inicio     = time.time()
    resultados = recuperar_via_grafo_ner(query, G_b1, k_hops=2, max_contextos=8)
    return resultados, (time.time() - inicio) * 1000


@observe(as_type="generation", name="B1_GraphRAG_Geracao")
def gerar_resposta_b1(query: str, contextos: List[Dict]) -> Tuple[str, float]:
    """
    Geração com LLM instrumentada como Generation no Langfuse.
    Registra tokens de prompt, tokens de resposta e custo estimado.
    """
    if contextos:
        evidencias_texto = "\n---\n".join([
            f"[{c['entidade_origem']} → {c['entidade_destino']}]: {c['contexto']}"
            for c in contextos
        ])
    else:
        evidencias_texto = "(Nenhuma evidência recuperada do grafo)"

    cadeia = PROMPT_RAG | modelo_llm
    inicio  = time.time()
    resposta = cadeia.invoke(
        {"evidencias": evidencias_texto, "query": query},
        config={"callbacks": [handler_langfuse]}
    )
    return resposta.content, (time.time() - inicio) * 1000


@observe(as_type="trace", name="B1_Pipeline_RAG_GraphNER")
def pipeline_b1(query: str, questao_id: str) -> Dict:
    """
    Pipeline RAG completo da Abordagem B1 instrumentado como Trace no Langfuse.
    Orquestra: Recuperação via Grafo NER → Geração com LLM.
    """
    contextos, lat_rec = recuperar_b1(query)
    resposta,  lat_ger = gerar_resposta_b1(query, contextos)
    return {
        "questao_id":              questao_id,
        "query":                   query,
        "resposta":                resposta,
        "contextos_recuperados":   contextos,
        "doc_ids_recuperados":     list({c["doc_id"] for c in contextos}),
        "num_contextos":           len(contextos),
        "latencia_recuperacao_ms": lat_rec,
        "latencia_geracao_ms":     lat_ger,
    }


# ── Executar pipeline ────────────────────────────────────────────────────
print(f"🚀 Iniciando pipeline B1 em {len(questoes)} questões...")
print(f"   Modelo: gpt-4o-mini | k_hops=2 | max_contextos=8\n")

resultados_pipeline, erros = [], []

for i, questao in enumerate(questoes):
    try:
        if (i + 1) % 10 == 0 or i == 0:
            print(f"   [{i+1}/{len(questoes)}] {questao['query'][:60]}...")

        resultado = pipeline_b1(questao["query"], questao["id"])
        resultado["ground_truth"]  = questao.get("ground_truth", "")
        resultado["golden_doc"]    = questao.get("golden_doc", [])
        resultado["tipo_questao"]  = questao.get("tipo", "?")
        resultado["dataset"]       = questao.get("dataset", "?")
        resultados_pipeline.append(resultado)
    except Exception as e:
        erros.append({"questao_id": questao["id"], "erro": str(e)})
        print(f"   ❌ Erro na questão {questao['id']}: {e}")

langfuse_client.flush()

print(f"\n✅ Concluído! Processadas: {len(resultados_pipeline)} | Erros: {len(erros)}")
if resultados_pipeline:
    lat_rec = [r["latencia_recuperacao_ms"] for r in resultados_pipeline]
    lat_ger = [r["latencia_geracao_ms"]     for r in resultados_pipeline]
    num_ctx = [r["num_contextos"]           for r in resultados_pipeline]
    print(f"   Latência recuperação: média={sum(lat_rec)/len(lat_rec):.0f}ms")
    print(f"   Latência geração:     média={sum(lat_ger)/len(lat_ger):.0f}ms")
    print(f"   Contextos por query:  média={sum(num_ctx)/len(num_ctx):.1f}")
    print(f"\n   Exemplo — Q: {resultados_pipeline[0]['query'][:60]}...")
    print(f"             R: {resultados_pipeline[0]['resposta'][:100]}...")


In [ ]:
# ============================================================
# CÉLULA 7 — Cálculo da Suite de Métricas
# ============================================================
# Métricas IR clássicas + métricas multi-hop (EM, Token F1).
# (DeepEval — Answer Relevancy, Faithfulness, Contextual trio —
#  pode ser adicionado mediante créditos de API DeepEval)
# ============================================================

import math
import re
import string


def calcular_ndcg(docs_recuperados: List[str], golden_docs: List[str], k: int = 5) -> float:
    """
    NDCG@K — Normalized Discounted Cumulative Gain.

    Premia documentos corretos nas posições mais altas do ranking.
    Diferente do P@K (que não considera posição), NDCG=1.0 indica ranking ideal.
    """
    golden_set = set(golden_docs)
    dcg  = sum(1.0 / math.log2(i + 2)
               for i, doc in enumerate(docs_recuperados[:k])
               if doc in golden_set)
    idcg = sum(1.0 / math.log2(i + 2)
               for i in range(min(len(golden_set), k)))
    return dcg / idcg if idcg > 0 else 0.0


def calcular_metricas_ir_classicas(
    docs_recuperados: List[str],
    golden_docs:      List[str],
    k_valores:        List[int] = [1, 3, 5, 10]
) -> dict:
    """
    Métricas clássicas de Information Retrieval baseadas em IDs de documentos.

    P@K  (Precision@K): proporção de docs no top-K que são relevantes
    R@K  (Recall@K):    proporção de docs relevantes recuperados no top-K
    F1@K:               média harmônica de P@K e R@K
    Hit@K:              1 se ao menos 1 golden_doc está no top-K
    MRR:                1 / posição do primeiro documento relevante
    NDCG@5:             DCG normalizado, sensível à posição
    """
    golden_set = set(golden_docs)
    resultados = {}

    for k in k_valores:
        top_k      = docs_recuperados[:k]
        relevantes = set(top_k) & golden_set
        p   = len(relevantes) / k           if k > 0        else 0.0
        r   = len(relevantes) / len(golden_set) if golden_set else 0.0
        f1  = 2 * p * r / (p + r)          if p + r > 0    else 0.0
        resultados[f"P@{k}"]   = round(p,   4)
        resultados[f"R@{k}"]   = round(r,   4)
        resultados[f"F1@{k}"]  = round(f1,  4)
        resultados[f"Hit@{k}"] = 1.0 if relevantes else 0.0

    mrr = 0.0
    for i, doc_id in enumerate(docs_recuperados, 1):
        if doc_id in golden_set:
            mrr = 1.0 / i
            break
    resultados["MRR"]    = round(mrr, 4)
    resultados["NDCG@5"] = round(calcular_ndcg(docs_recuperados, golden_docs), 4)
    return resultados


def normalizar_texto(texto: str) -> str:
    """Normalização padrão HotpotQA/2WikiMultiHopQA para EM e Token F1."""
    texto = texto.lower().strip()
    texto = re.sub(r'\b(a|an|the)\b', ' ', texto)
    texto = texto.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(texto.split())


def calcular_metricas_multihop(resposta: str, ground_truth: str) -> dict:
    """
    Exact Match (EM) e Token F1 — métricas padrão para QA multi-hop.

    EM:      rigoroso — respostas normalizadas devem ser idênticas
    Token F1: tolerante — mede sobreposição de tokens (útil para respostas longas)
    """
    r = normalizar_texto(resposta)
    g = normalizar_texto(ground_truth)
    em = 1.0 if r == g else 0.0

    r_t, g_t = set(r.split()), set(g.split())
    if not r_t or not g_t:
        f1 = 0.0
    else:
        comuns = r_t & g_t
        p      = len(comuns) / len(r_t)
        rec    = len(comuns) / len(g_t)
        f1     = 2 * p * rec / (p + rec) if p + rec > 0 else 0.0

    return {"EM": round(em, 4), "Token_F1": round(f1, 4)}


# ── Calcular métricas para todos os resultados ───────────────────────────
print("📐 Calculando métricas para todos os resultados...")

METRICAS_PARA_LANGFUSE = ["P@5", "R@5", "Hit@5", "MRR", "NDCG@5", "EM", "Token_F1"]

metricas_por_questao = []
for resultado in resultados_pipeline:
    mq = {
        "questao_id":              resultado["questao_id"],
        "tipo_questao":            resultado["tipo_questao"],
        "dataset":                 resultado["dataset"],
        "query":                   resultado["query"],
        "resposta":                resultado["resposta"],
        "ground_truth":            resultado["ground_truth"],
        "num_contextos":           resultado["num_contextos"],
        "latencia_recuperacao_ms": resultado["latencia_recuperacao_ms"],
        "latencia_geracao_ms":     resultado["latencia_geracao_ms"],
    }
    mq.update(calcular_metricas_ir_classicas(
        resultado["doc_ids_recuperados"], resultado["golden_doc"]
    ))
    mq.update(calcular_metricas_multihop(
        resultado["resposta"], resultado["ground_truth"]
    ))
    metricas_por_questao.append(mq)

df_metricas = pd.DataFrame(metricas_por_questao)

# ── Resumo agregado ──────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("MÉTRICAS AGREGADAS — Abordagem B1 (Knowledge Graph NER)")
print("=" * 60)

colunas = [c for c in METRICAS_PARA_LANGFUSE if c in df_metricas.columns]
medias  = df_metricas[colunas].mean()

print("\n📊 Médias sobre todas as questões:")
for metrica, valor in medias.items():
    barra = "█" * int(valor * 20)
    print(f"   {metrica:<12} {barra:<22} {valor:.4f}")

if "tipo_questao" in df_metricas.columns and df_metricas["tipo_questao"].nunique() > 1:
    chave = [m for m in ["P@5", "R@5", "MRR", "Token_F1"] if m in df_metricas.columns]
    print("\n📊 Médias por tipo de questão:")
    print(df_metricas.groupby("tipo_questao")[chave].mean().round(4).to_string())

print(f"\n💰 Custo de indexação: $0.00")
print(f"⏱️  Latência média recuperação: {df_metricas['latencia_recuperacao_ms'].mean():.0f}ms")
print(f"⏱️  Latência média geração:     {df_metricas['latencia_geracao_ms'].mean():.0f}ms")


In [ ]:
# ============================================================
# CÉLULA 8 — Envio de Scores ao Langfuse
# ============================================================
# Registra scores no dashboard Langfuse para comparação
# unificada entre todas as 6 abordagens (A1-A3, B1-B3).
# ============================================================

print("📡 Enviando scores ao Langfuse...")

scores_enviados = 0
erros_langfuse  = 0

for _, linha in df_metricas.iterrows():
    try:
        for metrica in METRICAS_PARA_LANGFUSE:
            if metrica in linha and pd.notna(linha[metrica]):
                langfuse_client.score(
                    name=f"B1_{metrica}",
                    value=float(linha[metrica]),
                    comment=f"Questão {linha['questao_id']} | Tipo: {linha.get('tipo_questao','?')}"
                )
                scores_enviados += 1
    except Exception:
        erros_langfuse += 1

# Médias globais como scores de sessão
colunas = [c for c in METRICAS_PARA_LANGFUSE if c in df_metricas.columns]
for metrica, valor in df_metricas[colunas].mean().items():
    try:
        langfuse_client.score(
            name=f"B1_MEDIA_{metrica}",
            value=float(valor),
            comment=f"Média sobre {len(df_metricas)} questões — Abordagem B1 GraphNER"
        )
    except Exception:
        pass

langfuse_client.flush()

print(f"✅ Scores enviados: {scores_enviados} | Erros: {erros_langfuse}")
print()
print("   📌 Para visualizar no Langfuse:")
print("      1. cloud.langfuse.com → Traces")
print("      2. Filtrar por: B1_Pipeline_RAG_GraphNER")
print("      3. Hierarquia por trace:")
print("         Trace → B1_GraphRAG_Recuperacao (Span)")
print("              → B1_GraphRAG_Geracao (Generation — tokens e custo)")


In [ ]:
# ============================================================
# CÉLULA 9 — Serialização dos Resultados
# ============================================================
# Persiste resultados no formato padronizado para uso no
# notebook 08_dashboard_comparativo.ipynb (comparação das 6 abordagens).
# Também salva o grafo NER para reutilização na Abordagem B3.
# ============================================================

CAMINHO_RESULTADOS = Path("resultados_B1.pkl")
CAMINHO_GRAFO      = Path("grafo_B1.pkl")
colunas_m          = [c for c in METRICAS_PARA_LANGFUSE if c in df_metricas.columns]

dados_para_salvar = {
    # Metadados da abordagem
    "abordagem":   "B1",
    "descricao":   "Knowledge Graph via NER puro (co-ocorrência, sem ontologia)",
    "modelo_ner":  "en_core_web_trf",
    "modelo_llm":  "gpt-4o-mini",
    "k_hops":      2,
    "max_contextos": 8,
    # Resultados do pipeline
    "resultados_pipeline":  resultados_pipeline,
    "df_metricas":          df_metricas,
    "metricas_agregadas":   df_metricas[colunas_m].mean().to_dict(),
    # Estatísticas do grafo
    "stats_grafo": {
        "num_nos":    G_b1.number_of_nodes(),
        "num_arestas": G_b1.number_of_edges(),
        "densidade":  nx.density(G_b1),
    },
    # Custos e latências
    "custo_indexacao_usd":           0.0,
    "tempo_indexacao_s":             tempo_indexacao_s,
    "latencia_media_recuperacao_ms": df_metricas["latencia_recuperacao_ms"].mean() if len(df_metricas) > 0 else 0,
    "latencia_media_geracao_ms":     df_metricas["latencia_geracao_ms"].mean()     if len(df_metricas) > 0 else 0,
}

# Salvar resultados
with open(CAMINHO_RESULTADOS, "wb") as f:
    pickle.dump(dados_para_salvar, f)

# Salvar grafo para reutilização na Abordagem B3
with open(CAMINHO_GRAFO, "wb") as f:
    pickle.dump(G_b1, f)

print(f"✅ Resultados salvos em: {CAMINHO_RESULTADOS}")
print(f"✅ Grafo NER salvo em:   {CAMINHO_GRAFO}")
print(f"   (grafo_B1.pkl será reutilizado na B3 — detecção de comunidades semânticas)")
print()
print(f"   Métricas agregadas B1:")
for metrica, valor in dados_para_salvar["metricas_agregadas"].items():
    print(f"     {metrica}: {valor:.4f}")
print()
print("   Para carregar no dashboard:")
print("   >>> import pickle")
print("   >>> with open('resultados_B1.pkl', 'rb') as f: dados = pickle.load(f)")


## 📈 Análise dos Resultados — Abordagem B1

### O que acabamos de observar?

#### Limitações Estruturais da Co-ocorrência Pura

**1. Contextual Precision tende a ser baixa (Hipótese H4)**

O principal problema do B1 é que a aresta `CO_OCORRE_COM` não carrega semântica:

```
"James Cameron won the Oscar in 1998"
  → Cameron --CO_OCORRE_COM--> Oscar     ✅ (relação real: premiação)
  → Cameron --CO_OCORRE_COM--> 1998      ❓ (relação vaga: co-ocorrência)

"The film Titanic grossed $2 billion in 1998"
  → Titanic --CO_OCORRE_COM--> 1998      ← RUÍDO (mesmo aresta, outro contexto)
```

O grafo não distingue as duas situações — ambas geram a mesma aresta `CO_OCORRE_COM`.
Resultado: ao buscar "quem ganhou o Oscar em 1998?", o traversal devolve tudo que co-ocorre
com "Oscar" e "1998", incluindo contextos irrelevantes.

**2. Sensibilidade ao vocabulário da query**

O NER precisa reconhecer na query os mesmos tokens presentes no grafo:
- `"Who directed Titanic?"` → NER: "Titanic" (WORK_OF_ART) ✅
- `"Which 1997 film won the most Oscars?"` → NER: "1997" (DATE), "Oscars" (ORG) ⚠️

**3. Grafos esparsos para corpora pequenos**

Com poucos documentos, muitas entidades aparecem apenas uma vez, gerando componentes isolados
com baixa capacidade de traversal multi-hop.

#### Vantagens do B1

**1. Custo zero de indexação**
Não requer chamada de API — spaCy roda localmente.
Para corpora com milhares de documentos, isso representa economia real vs. B2 (~$0.02).

**2. Multi-hop estruturalmente suportado**
O traversal de grafo conecta entidades via intermediária:
```
"James Cameron" → Titanic → Oscar → 1998 Academy Awards
```
Difícil de capturar com chunking fixo (A1), onde a informação pode estar em chunks diferentes.

**3. Explicabilidade do caminho**
O caminho no grafo é auditável — é possível mostrar ao usuário a cadeia de entidades
que levou ao contexto recuperado.

### Comparação com o que vem a seguir

| Abordagem | Tipo de Relação | Custo Indexação | Precisão Esperada |
|-----------|----------------|-----------------|-------------------|
| **B1 (este)** | CO_OCORRE_COM (sem semântica) | **$0.00** | Baixa-Moderada |
| **B2 (próximo)** | CRIOU, DIRIGIU, PREMIADO_COM... | ~$0.02 | Alta |
| **B3 (depois)** | Comunidades semânticas (Louvain) | ~$0.024 | Alta (global) |

### Verificação da Hipótese H4

> **H4**: "KG sem ontologia (B1) tem baixa Contextual Precision?"

Compare `P@5` e `MRR` do B1 com os resultados do B2 após executar o próximo notebook.
A hipótese prevê B1 < B2, pois as arestas de co-ocorrência incluem muitos falsos positivos.

### Próximo Passo

Execute `06_B2_kg_ontologia.ipynb` para implementar a Abordagem B2, onde a extração é
guiada por uma ontologia pré-definida com relações semanticamente ricas:
`CRIOU`, `DIRIGIU`, `PREMIADO_COM`, `NASCEU_EM`, `OCORREU_EM`...